# Attention Score Decomposition

Decompose pre-softmax attention scores: magnitude profiles, position
decomposition, temperature analysis, cross-head comparison, and saturation.

## Why This Matters

Composition attention score decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_score_decomposition import (
    score_magnitude_profile, score_position_decomposition,
    score_temperature_analysis, score_cross_head_comparison,
    score_softmax_saturation,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Score Magnitude Profile

What range of pre-softmax scores does each head produce?

In [ ]:
result = score_magnitude_profile(model, tokens)
print(f"Heads analyzed: {len(result['per_head'])}\n")
for h in result['per_head']:
    bar = '█' * int(min(h['score_range'], 5) * 4)
    print(f"  L{h['layer']}H{h['head']}: range={h['score_range']:.3f} "
          f"[{h['min_score']:.3f}, {h['max_score']:.3f}] "
          f"mean={h['mean_score']:.3f} {bar}")

## Score Position Decomposition

For a specific query position, break down the score to each key position
into query norm, key norm, and cosine components.

In [ ]:
result = score_position_decomposition(model, tokens, layer=0, head=0, position=-1)
print(f"Query norm: {result['query_norm']:.4f}\n")
for k in result['per_key']:
    print(f"  Key pos {k['position']}: score={k['score']:.4f}, "
          f"key_norm={k['key_norm']:.4f}, cos={k['cosine']:.4f}")

## Temperature Analysis

Effective temperature and entropy of the attention distribution per position.

In [ ]:
result = score_temperature_analysis(model, tokens, layer=0, head=0)
for p in result['per_position']:
    bar = '█' * int(min(p['effective_temperature'], 5) * 4)
    print(f"  Pos {p['position']}: temp={p['effective_temperature']:.4f}, "
          f"max_prob={p['max_probability']:.4f}, H={p['entropy']:.4f} {bar}")

## Cross-Head Comparison

How similar are the score patterns between different heads in a layer?

In [ ]:
result = score_cross_head_comparison(model, tokens, layer=0)
print(f"Head pairs: {len(result['pairs'])}\n")
for p in result['pairs']:
    print(f"  H{p['head_a']} vs H{p['head_b']}: "
          f"score_sim={p['score_similarity']:.4f}, "
          f"pattern_sim={p['pattern_similarity']:.4f}")

## Softmax Saturation

Are any heads saturated (placing nearly all probability mass on one key)?

In [ ]:
result = score_softmax_saturation(model, tokens)
print(f"Saturated heads: {result['n_saturated']}/{len(result['per_head'])}\n")
for h in result['per_head']:
    sat = ' [SATURATED]' if h['is_saturated'] else ''
    print(f"  L{h['layer']}H{h['head']}: max_prob={h['mean_max_prob']:.4f}, "
          f"score_range={h['mean_score_range']:.4f}{sat}")